# Bond Portfolio Risk Engine — Colab Setup

This notebook initializes a reproducible Google Colab workspace for the project. It does not calculate VaR, Expected Shortfall, or stress losses.

Run the cells from top to bottom. GitHub is the source of truth; the Colab runtime under `/content` is temporary.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
print(f'Python: {platform.python_version()}')

## Clone or update the repository

The public repository can be cloned without a GitHub credential. If the runtime already contains the repository, the cell performs a fast-forward-only update.

In [ ]:
REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_DIR = Path('/content/market_risk') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_DIR)
src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Project directory: {PROJECT_DIR}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

## Install declared dependencies

Colab runtimes change over time. Installing the repository requirements makes the environment explicit, while the version report below records the resolved versions. Restart the runtime only if Colab requests it after installation.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements-colab.txt'],
    check=True,
)

## Verify the environment and project contract

In [ ]:
from importlib.metadata import version

packages = ['numpy', 'pandas', 'scipy', 'matplotlib', 'plotly', 'statsmodels', 'requests', 'PyYAML', 'pyarrow']
for package in packages:
    print(f'{package}: {version(package)}')

required_files = [
    'docs/WEEK01_RESEARCH_SPEC.md',
    'docs/DATA_DICTIONARY.md',
    'docs/WEEK01_ACCEPTANCE.md',
    'docs/MILESTONES.md',
]
missing = [path for path in required_files if not (PROJECT_DIR / path).is_file()]
assert not missing, f'Missing required project files: {missing}'
print('Project contract: OK')

## Optional: check the FRED API secret

Add `FRED_API_KEY` through the Colab Secrets panel. This cell reports availability without printing the secret. The data-ingestion milestone will consume it.

In [ ]:
FRED_API_KEY = None
if IN_COLAB:
    try:
        from google.colab import userdata
        FRED_API_KEY = userdata.get('FRED_API_KEY')
    except Exception:
        FRED_API_KEY = None

print(f'FRED_API_KEY available: {bool(FRED_API_KEY)}')

## Next milestone

M02 will add a versioned Federal Reserve GSW/FRED ingestion and data-audit pipeline. Reusable implementation will live under `src/`; notebooks will call that implementation and present reviewable outputs.